In [1]:
import os
import dashscope

dashscope.api_key = os.environ["DASHSCOPE_API_KEY"]

response = dashscope.TextEmbedding.call(
    model="qwen3.7-text-embedding",
    input=[
        "向量数据库是 RAG 系统的重要基础设施。",
        "Embedding 可以将文本转换为向量。",
    ],
)

if response.status_code != 200:
    raise RuntimeError(f"{response.code}: {response.message}")

for item in response.output["embeddings"]:
    vector = item["embedding"]
    print(f"索引：{item['text_index']}")
    print(f"向量维度：{len(vector)}")
    print(f"前 5 个值：{vector[:5]}")

索引：0
向量维度：1024
前 5 个值：[0.032408494502305984, -0.023811100050807, 0.00760685233399272, 0.014769230969250202, -0.03436417877674103]
索引：1
向量维度：1024
前 5 个值：[0.022215088829398155, 0.01910962164402008, 0.023352129384875298, -0.0006094019045121968, 0.002118197036907077]


In [2]:
from pathlib import Path

WORK_DIR = Path.cwd()
KNOWLEDGE_DIR = WORK_DIR / "knowledge_base"
DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "milvus_qwen_demo.db"

if not KNOWLEDGE_DIR.exists():
    raise FileNotFoundError(
        f"没有找到 {KNOWLEDGE_DIR}。请把当前工作目录切换到 Notebook 所在目录。"
    )

docx_files = sorted(KNOWLEDGE_DIR.glob("*.docx"))
if not docx_files:
    raise FileNotFoundError(f"{KNOWLEDGE_DIR} 中没有 DOCX 文件")

print("工作目录：", WORK_DIR)
print("数据库文件：", DB_PATH)
print(f"发现 {len(docx_files)} 份 DOCX：")
for path in docx_files:
    print("-", path.name)


工作目录： /Users/jackhu/src_code/task-pivot/ai-agent
数据库文件： /Users/jackhu/src_code/task-pivot/ai-agent/data/milvus_qwen_demo.db
发现 6 份 DOCX：
- 01_员工休假制度.docx
- 02_费用报销制度.docx
- 03_远程办公制度.docx
- 04_办公区域安全制度.docx
- 05_线上故障响应制度.docx
- 06_员工培训制度.docx


In [3]:
from collections import defaultdict
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode
from llama_index.readers.file import DocxReader

documents = SimpleDirectoryReader(
    input_dir=str(KNOWLEDGE_DIR),
    required_exts=[".docx"],
    file_extractor={".docx": DocxReader()},
    recursive=False,
    raise_on_error=True,
).load_data()

splitter = SentenceSplitter(
    chunk_size=350,
    chunk_overlap=50,
)
nodes = splitter.get_nodes_from_documents(documents)

chunk_counter = defaultdict(int)
chunks = []

for node in nodes:
    source_file = node.metadata.get("file_name")
    if not source_file and node.metadata.get("file_path"):
        source_file = Path(node.metadata["file_path"]).name

    text = node.get_content(metadata_mode=MetadataMode.NONE).strip()
    if not text:
        continue

    chunk_index = chunk_counter[source_file]
    chunk_counter[source_file] += 1

    chunks.append(
        {
            "text": text,
            "source_file": source_file,
            "chunk_index": chunk_index,
        }
    )

print(f"Document 数量：{len(documents)}")
print(f"Chunk 数量：{len(chunks)}")
print("第一条 Chunk：")
print(chunks[0])


Document 数量：6
Chunk 数量：12
第一条 Chunk：
{'text': '示例科技有限公司｜内部制度文件\n\n员工休假制度\n\n企业内部管理制度\n\n归口部门：人力资源部\n\n适用范围：中国大陆地区正式员工\n\n生效日期：2026年1月1日\n\n文件版本：V1.0\n\n\n\n一、制度目的\n\n规范员工带薪年假的申请、审批与结转管理，保障员工休息权益并维持团队工作的连续性。\n\n二、制度内容\n\n第一条 年假额度\n\n正式员工每个自然年度享有10个工作日的带薪年假。新入职员工当年度的年假额度按照实际在职月份折算。\n\n年假以工作日为计算单位，国家法定节假日和周末不计入已使用的年假天数。\n\n第二条 申请与审批\n\n员工应至少提前3个工作日在OA系统提交年假申请，并由直属主管审批。', 'source_file': '01_员工休假制度.docx', 'chunk_index': 0}


In [6]:
import dis

import dashscope
import os

EMBEDDING_MODEL = "qwen3.7-text-embedding"
dashscope.api_key = os.environ["DASHSCOPE_API_KEY"]

def embed_texts(texts: list[str], batch_size: int = 8) -> list[list[float]]:
    # 批量调用模型，并保持输出顺序与输入一致。
    if not texts:
        return []

    all_vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        response = dashscope.TextEmbedding.call(
            model=EMBEDDING_MODEL,
            input=batch,
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"Embedding 请求失败：{response.code} - {response.message}"
            )

        ordered_items = sorted(
            response.output["embeddings"],
            key=lambda item: item["text_index"],
        )
        all_vectors.extend(item["embedding"] for item in ordered_items)

    if len(all_vectors) != len(texts):
        raise RuntimeError("Embedding 数量与输入文本数量不一致")

    return all_vectors

dimension_probe = embed_texts(["这是用于检测向量维度的文本"])[0]
EMBEDDING_DIM = len(dimension_probe)

print("Embedding 模型：", EMBEDDING_MODEL)
print("向量维度：", EMBEDDING_DIM)
print("前 5 个值：", dimension_probe[:5])

Embedding 模型： qwen3.7-text-embedding
向量维度： 1024
前 5 个值： [0.027331871911883354, 0.03176603093743324, -0.0005497268284671009, -0.0062332660891115665, -0.020389866083860397]


In [8]:
chunk_texts = [item["text"] for item in chunks]
for c in chunk_texts:
    print(c)

chunk_vectors = embed_texts(chunk_texts, batch_size=8)

assert len(chunk_vectors) == len(chunks)
assert all(len(vector) == EMBEDDING_DIM for vector in chunk_vectors)

print(f"已经为 {len(chunk_vectors)} 个 Chunk 生成向量")

示例科技有限公司｜内部制度文件

员工休假制度

企业内部管理制度

归口部门：人力资源部

适用范围：中国大陆地区正式员工

生效日期：2026年1月1日

文件版本：V1.0



一、制度目的

规范员工带薪年假的申请、审批与结转管理，保障员工休息权益并维持团队工作的连续性。

二、制度内容

第一条 年假额度

正式员工每个自然年度享有10个工作日的带薪年假。新入职员工当年度的年假额度按照实际在职月份折算。

年假以工作日为计算单位，国家法定节假日和周末不计入已使用的年假天数。

第二条 申请与审批

员工应至少提前3个工作日在OA系统提交年假申请，并由直属主管审批。
第二条 申请与审批

员工应至少提前3个工作日在OA系统提交年假申请，并由直属主管审批。连续休假超过5个工作日时，还需由部门负责人确认工作交接安排。

第三条 年假结转

未使用的年假最多可以结转5天至下一年度，结转年假应在下一年度6月30日前使用完毕，逾期不再保留。

内部资料，请妥善保管｜第 1 页
示例科技有限公司｜内部制度文件

费用报销制度

企业内部管理制度

归口部门：财务部

适用范围：全体员工

生效日期：2026年1月1日

文件版本：V1.0
一、制度目的

统一业务费用的申请、凭证和审批要求，确保公司费用真实、合规并可追溯。

二、制度内容

第一条 提交时限

员工应在费用发生后的5个工作日内提交报销申请，并附上合法有效的发票及必要的业务说明。超过时限的申请应说明原因。

第二条 审批权限

普通费用由直属主管审批。单笔金额超过5000元的费用还需要部门负责人审批，超过预算的费用须事先取得财务部确认。

第三条 支付与归档

财务部完成合规审核后统一安排付款。电子申请记录和原始凭证按照公司档案管理要求保存。

内部资料，请妥善保管｜第 1 页
示例科技有限公司｜内部制度文件

远程办公制度

企业内部管理制度

归口部门：人力资源部、信息技术部

适用范围：通过试用期且岗位适合远程协作的员工

生效日期：2026年1月1日

文件版本：V1.0
一、制度目的

明确远程办公的适用时间、申请方式和信息安全要求，保证远程协作效率。

二、制度内容

第一条 适用时间

符合条件的员工可以在每周三申请远程办公。遇到公司级会议、客户现场工作或团队集中协作安排时，应优先服从现场